In [1]:
import os
import json
import glob
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

In [2]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [3]:
BASEDIR = '/Users/egyptdj/github/bruit-classification'
DATADIR = os.path.join(BASEDIR, 'data_v5')
MODELDIR = os.path.join(BASEDIR, 'onestep_model')

MODEL_HANDLE = 'https://tfhub.dev/google/yamnet/1'
SAMPLE_RATE = 16000
BATCH_SIZE = 1
SEED = 0

In [4]:
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [5]:
train_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'train', '**/*.wav')), dtype='object')
train_audio_label_list = [f.split('/')[-2] for f in train_audio_file_list]

In [6]:
val_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'val', '**/*.wav')), dtype='object')
val_audio_label_list = [f.split('/')[-2] for f in val_audio_file_list]

In [7]:
test_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'test', '**/*.wav')), dtype='object')
test_audio_label_list = [f.split('/')[-2] for f in test_audio_file_list]

In [8]:
class_name_dict = {}
i = 0
for label in train_audio_label_list:
  if not label in list(class_name_dict.keys()):
    class_name_dict[label] = i
    i += 1

In [9]:
with open(os.path.join(BASEDIR, 'class_name_dict.json'), 'w') as f:
  json.dump(class_name_dict, f)

In [10]:
train_audio_label_list = pd.Series([class_name_dict[i] for i in train_audio_label_list], dtype='int64')
val_audio_label_list = pd.Series([class_name_dict[i] for i in val_audio_label_list], dtype='int64')
test_audio_label_list = pd.Series([class_name_dict[i] for i in test_audio_label_list], dtype='int64')

In [11]:
train_audio_label_list = train_audio_label_list.replace({class_name_dict['background']: 0, class_name_dict['normal']: 1, class_name_dict['bruit']: 2, class_name_dict['normal-noise']: 3, class_name_dict['bruit-noise']: 3})
val_audio_label_list = val_audio_label_list.replace({class_name_dict['background']: 0, class_name_dict['normal']: 1, class_name_dict['bruit']: 2, class_name_dict['normal-noise']: 3, class_name_dict['bruit-noise']: 3})
test_audio_label_list = test_audio_label_list.replace({class_name_dict['background']: 0, class_name_dict['normal']: 1, class_name_dict['bruit']: 2, class_name_dict['normal-noise']: 3, class_name_dict['bruit-noise']: 3})

In [12]:
train_ds = tf.data.Dataset.from_tensor_slices((train_audio_file_list, train_audio_label_list))
val_ds = tf.data.Dataset.from_tensor_slices((val_audio_file_list, val_audio_label_list))
test_ds = tf.data.Dataset.from_tensor_slices((test_audio_file_list, test_audio_label_list))

In [13]:
# *.wav 파일을 경로에서부터 오디오파일로 읽어오기

@tf.function
def load_wav_and_label(audio_file, label):
  audio = tf.io.read_file(audio_file)
  audio, sample_rate = tf.audio.decode_wav(audio)
  audio = tf.squeeze(audio, axis=-1)
  return audio, label

In [14]:
yamnet_model = hub.load(MODEL_HANDLE)

In [15]:
@tf.function
def extract_embedding(wav_data, label):
  scores, embeddings, spectrogram = yamnet_model(wav_data)
  num_embeddings = tf.shape(embeddings)[0]
  return embeddings, tf.repeat(label, num_embeddings)

In [16]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024), dtype=tf.float32, name='yamnet_embedding'),
    tf.keras.layers.Dense(4)
])

In [17]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 4)                 4100      
                                                                 
Total params: 4,100
Trainable params: 4,100
Non-trainable params: 0
_________________________________________________________________


In [18]:
model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(
                  learning_rate=1e-5,
              ),
              metrics=['accuracy'],
)

In [19]:
callback = tf.keras.callbacks.EarlyStopping(
    monitor='loss',
    patience=3,
    restore_best_weights=True
)

In [20]:
train_ds = train_ds.map(load_wav_and_label).map(extract_embedding).cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(load_wav_and_label).map(extract_embedding).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(load_wav_and_label).map(extract_embedding).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [21]:
history = model.fit(train_ds,
                    epochs=100,
                    validation_data=val_ds,
                    callbacks=callback
                    )

Epoch 1/100
985/985 [==============================] - 38s 3ms/step - loss: 1.3875 - accuracy: 0.2376 - val_loss: 1.3439 - val_accuracy: 0.3161
Epoch 2/100
985/985 [==============================] - 3s 3ms/step - loss: 1.3475 - accuracy: 0.2862 - val_loss: 1.3118 - val_accuracy: 0.3585
Epoch 3/100
985/985 [==============================] - 3s 3ms/step - loss: 1.3134 - accuracy: 0.3253 - val_loss: 1.2836 - val_accuracy: 0.3889
Epoch 4/100
985/985 [==============================] - 3s 3ms/step - loss: 1.2830 - accuracy: 0.3568 - val_loss: 1.2568 - val_accuracy: 0.4074
Epoch 5/100
985/985 [==============================] - 3s 3ms/step - loss: 1.2551 - accuracy: 0.3789 - val_loss: 1.2314 - val_accuracy: 0.4259
Epoch 6/100
985/985 [==============================] - 3s 3ms/step - loss: 1.2294 - accuracy: 0.3981 - val_loss: 1.2094 - val_accuracy: 0.4418
Epoch 7/100
985/985 [==============================] - 3s 3ms/step - loss: 1.2058 - accuracy: 0.4139 - val_loss: 1.1884 - val_accuracy: 0.466

In [22]:
model.evaluate(test_ds)

68/68 [==============================] - 3s 38ms/step - loss: 0.8335 - accuracy: 0.6392


[0.8334841728210449, 0.6391863226890564]

In [23]:
input_segment = tf.keras.layers.Input(shape=(), dtype=tf.float32, name='audio')
embedding_extraction_layer = hub.KerasLayer(MODEL_HANDLE,
                                            trainable=False, name='yamnet')
_, embeddings, _ = embedding_extraction_layer(input_segment)
final_output = tf.math.reduce_mean(model(embeddings), axis=0)

In [24]:
serving_model = tf.keras.Model(input_segment, final_output)

In [25]:
serving_model.save(MODELDIR, include_optimizer=False)

INFO:tensorflow:Assets written to: /Users/egyptdj/github/bruit-classification/onestep_model/assets


INFO:tensorflow:Assets written to: /Users/egyptdj/github/bruit-classification/onestep_model/assets


In [26]:
converter = tf.lite.TFLiteConverter.from_saved_model(MODELDIR)

In [27]:
tflite_model = converter.convert()

In [28]:
with open(os.path.join(MODELDIR, 'bruit_yamnet_onestep.tflite'), 'wb') as f:
  f.write(tflite_model)